In [1]:
import gi
gi.require_version('Gtk', '3.0')
from gi.repository import Gtk, GLib, Gdk, Pango

import threading
import os
import re
import socket
import json
import traceback

class ChatbotWindow(Gtk.Window):
    def __init__(self):
        Gtk.Window.__init__(self, title="RAG Chatbot")
        self.set_default_size(500, 400)
        self.set_border_width(10)

        self.apply_css()
        
        # Socket connection
        self.host = '127.0.0.1'
        self.port = 65432
        self.client_socket = None
        self.is_connected = False

        vbox = Gtk.Box(orientation=Gtk.Orientation.VERTICAL, spacing=10)
        self.add(vbox)

        action_bar = Gtk.Box(orientation=Gtk.Orientation.HORIZONTAL, spacing=5)
        vbox.pack_start(action_bar, False, False, 0)

        clear_button = Gtk.Button.new_from_icon_name("edit-clear-all-symbolic", Gtk.IconSize.MENU)
        clear_button.set_tooltip_text("Clear conversation")
        clear_button.connect("clicked", self.on_clear_button_clicked)
        action_bar.pack_start(clear_button, False, False, 0)
        
        save_button = Gtk.Button.new_from_icon_name("document-save-symbolic", Gtk.IconSize.MENU)
        save_button.set_tooltip_text("Save conversation")
        save_button.connect("clicked", self.on_save_button_clicked)
        action_bar.pack_start(save_button, False, False, 0)
        
        file_button = Gtk.Button.new_from_icon_name("document-open-symbolic", Gtk.IconSize.MENU)
        file_button.set_tooltip_text("Select Knowledge Base (PDF)")
        file_button.connect("clicked", self.on_file_button_clicked)
        action_bar.pack_start(file_button, False, False, 0)

        scrolled_window = Gtk.ScrolledWindow()
        scrolled_window.set_hexpand(True)
        scrolled_window.set_vexpand(True)
        vbox.pack_start(scrolled_window, True, True, 0)
        scrolled_window.get_style_context().add_class("chat-history")

        self.message_view = Gtk.TextView()
        self.message_view.set_editable(False)
        self.message_view.set_cursor_visible(False)
        self.message_view.set_left_margin(10)
        self.message_view.set_right_margin(10)
        self.message_view.set_wrap_mode(Gtk.WrapMode.WORD)
        scrolled_window.add(self.message_view)

        self.message_buffer = self.message_view.get_buffer()
        self.tag_user = self.message_buffer.create_tag(
            "user", 
            foreground="#333333", 
            weight=600, 
            justification=Gtk.Justification.LEFT
        )
        self.tag_chatbot = self.message_buffer.create_tag(
            "chatbot", 
            foreground="#00796b", 
            weight=400, 
            justification=Gtk.Justification.LEFT
        )
        self.tag_system = self.message_buffer.create_tag(
            "system", 
            foreground="#999999", 
            style=Pango.Style.ITALIC, 
            justification=Gtk.Justification.CENTER
        )
        
        input_box = Gtk.Box(orientation=Gtk.Orientation.HORIZONTAL, spacing=6)
        vbox.pack_start(input_box, False, False, 0)

        self.input_entry = Gtk.Entry()
        self.input_entry.set_hexpand(True)
        self.input_entry.get_style_context().add_class("input-entry")
        input_box.pack_start(self.input_entry, True, True, 0)
        self.input_entry.connect("activate", self.on_send_button_clicked)

        self.send_button = Gtk.Button.new_from_icon_name("mail-send-receive-symbolic", Gtk.IconSize.MENU)
        self.send_button.set_tooltip_text("Send message")
        self.send_button.connect("clicked", self.on_send_button_clicked)
        input_box.pack_start(self.send_button, False, False, 0)

        self.spinner = Gtk.Spinner()
        input_box.pack_start(self.spinner, False, False, 0)

        self.append_message("Attempting to connect to the server...", "system")
        self.connect_to_server()

    def connect_to_server(self):
        try:
            self.client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.client_socket.connect((self.host, self.port))
            self.is_connected = True
            GLib.idle_add(self.on_connection_successful)
        except ConnectionRefusedError:
            GLib.idle_add(self.on_connection_failed)
        except Exception as e:
            GLib.idle_add(self.on_connection_failed, e)
    
    def on_connection_successful(self):
        self.append_message("Connected to server. Select a PDF to begin.", "system")
        self.input_entry.set_sensitive(False)
        self.send_button.set_sensitive(False)
    
    def on_connection_failed(self, error=None):
        self.append_message(f"Could not connect to server. Please ensure the server is running. Error: {error}", "system")
        self.input_entry.set_sensitive(False)
        self.send_button.set_sensitive(False)

    def send_request(self, request):
        if not self.is_connected:
            GLib.idle_add(self.append_message, "Not connected to server.", "system")
            return None
        try:
            self.client_socket.sendall(json.dumps(request).encode('utf-8'))
            response_data = self.client_socket.recv(4096)
            return json.loads(response_data.decode('utf-8'))
        except (socket.error, json.JSONDecodeError) as e:
            GLib.idle_add(self.append_message, f"Communication error: {e}", "system")
            self.is_connected = False
            return None

    def on_file_button_clicked(self, widget):
        if not self.is_connected:
            self.append_message("Cannot select a file. Not connected to server.", "system")
            return
            
        dialog = Gtk.FileChooserDialog(
            "Please choose a PDF file", self, Gtk.FileChooserAction.OPEN,
            (Gtk.STOCK_CANCEL, Gtk.ResponseType.CANCEL, "Select", Gtk.ResponseType.ACCEPT)
        )
        
        filter_pdf = Gtk.FileFilter()
        filter_pdf.set_name("PDF files")
        filter_pdf.add_mime_type("application/pdf")
        dialog.add_filter(filter_pdf)
        
        response = dialog.run()
        if response == Gtk.ResponseType.ACCEPT:
            pdf_path = dialog.get_filename()
            self.input_entry.set_sensitive(False)
            self.send_button.set_sensitive(False)
            self.append_message(f"Loading {os.path.basename(pdf_path)}...", "system")
            self.spinner.start()
            
            thread = threading.Thread(target=self.send_init_request, args=(pdf_path,))
            thread.daemon = True
            thread.start()
            
        dialog.destroy()

    def send_init_request(self, pdf_path):
        request = {"type": "init_pdf", "file_path": pdf_path}
        response = self.send_request(request)
        GLib.idle_add(self.on_response_received, response)

    def on_send_button_clicked(self, widget):
        if not self.is_connected:
            self.append_message("Cannot send message. Not connected to server.", "system")
            return

        user_text = self.input_entry.get_text()
        if not user_text:
            return

        self.append_message(user_text, "user")
        self.input_entry.set_text("")
        self.input_entry.set_sensitive(False)
        self.send_button.set_sensitive(False)
        self.spinner.start()

        thread = threading.Thread(target=self.send_query_request, args=(user_text,))
        thread.daemon = True
        thread.start()

    def send_query_request(self, user_text):
        request = {"type": "query", "text": user_text}
        response = self.send_request(request)
        GLib.idle_add(self.on_response_received, response)

    def on_response_received(self, response):
        if not response:
            self.append_message("No response received from server.", "system")
            self.spinner.stop()
            self.input_entry.set_sensitive(self.is_connected)
            self.send_button.set_sensitive(self.is_connected)
            return

        if response['status'] == 'success':
            if 'response' in response:
                formatted_response = self.format_response(response['response'])
                self.append_message(formatted_response, "chatbot")
            else:
                self.append_message(response['message'], "system")
        else:
            self.append_message(f"Server error: {response['message']}", "system")

        self.spinner.stop()
        self.input_entry.set_sensitive(self.is_connected)
        self.send_button.set_sensitive(self.is_connected)

    def format_response(self, text):
        text = re.sub(r'\n{2,}', '\n', text)
        text = re.sub(r'(\d+\.)\s', r'\n\1 ', text)
        text = re.sub(r'([-*])\s', r'\n\1 ', text)
        text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text) 
        
        return text

    def on_clear_button_clicked(self, widget):
        self.message_buffer.set_text("")
        self.append_message("Chat history cleared.", "system")

    def on_save_button_clicked(self, widget):
        dialog = Gtk.FileChooserDialog(
            "Save Conversation", self, Gtk.FileChooserAction.SAVE,
            (Gtk.STOCK_CANCEL, Gtk.ResponseType.CANCEL, "Save", Gtk.ResponseType.ACCEPT)
        )
        dialog.set_current_name("chatbot_conversation.txt")

        response = dialog.run()
        if response == Gtk.ResponseType.ACCEPT:
            filename = dialog.get_filename()
            with open(filename, "w") as f:
                start_iter, end_iter = self.message_buffer.get_bounds()
                text = self.message_buffer.get_text(start_iter, end_iter, True)
                f.write(text)
        
        dialog.destroy()

    def append_message(self, message, tag_name):
        end_iter = self.message_buffer.get_end_iter()
        start_iter = self.message_buffer.get_end_iter()
        
        if tag_name == "user":
            self.message_buffer.insert(end_iter, "You: ")
        elif tag_name == "chatbot":
            self.message_buffer.insert(end_iter, "Chatbot: ")
        elif tag_name == "system":
            self.message_buffer.insert(end_iter, "System: ")

        self.message_buffer.insert(end_iter, message + "\n")
        
        end_iter = self.message_buffer.get_end_iter()
        self.message_buffer.apply_tag_by_name(tag_name, start_iter, end_iter)
        
        self.message_view.scroll_to_iter(end_iter, 0.0, True, 0.0, 1.0)
        return False

    def apply_css(self):
        style_provider = Gtk.CssProvider()
        try:
            style_provider.load_from_path('style.css')
            Gtk.StyleContext.add_provider_for_screen(
                Gdk.Screen.get_default(),
                style_provider,
                Gtk.STYLE_PROVIDER_PRIORITY_APPLICATION
            )
        except Exception as e:
            print(f"Failed to load CSS: {e}")

if __name__ == "__main__":
    win = ChatbotWindow()
    win.connect("destroy", Gtk.main_quit)
    win.show_all()
    Gtk.main()


(ipykernel_launcher.py:35303): Gtk-WARNING **: 10:46:29.404: Invalid text buffer iterator: either the iterator is uninitialized, or the characters/pixbufs/widgets in the buffer have been modified since the iterator was created.
You must use marks, character numbers, or line numbers to preserve a position across buffer modifications.
You can apply tags and insert marks without invalidating your iterators,
but any mutation that affects 'indexable' buffer contents (contents that can be referred to by character offset)
will invalidate all outstanding iterators

(ipykernel_launcher.py:35303): Gtk-CRITICAL **: 10:46:29.404: gtk_text_buffer_apply_tag_by_name: assertion 'gtk_text_iter_get_buffer (start) == buffer' failed

(ipykernel_launcher.py:35303): Gtk-WARNING **: 10:46:29.426: Invalid text buffer iterator: either the iterator is uninitialized, or the characters/pixbufs/widgets in the buffer have been modified since the iterator was created.
You must use marks, character numbers, or line 